In [ ]:
### Geolibraries
import geopandas as gpd
import osmnx as ox
import contextily as ctx


# General tools
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pyarrow.parquet as pq

from h3 import h3
from shapely.geometry import Polygon

In [ ]:
users = pd.read_parquet("./data/user_pois_pt_1.parquet")

In [ ]:
home_users_df = (
    users[users["is_home"] == 1]
    .drop_duplicates(subset="user_id")
)

In [ ]:
home_users_df

In [ ]:
#bring the POI layer
pois = pd.read_parquet("data/pois_per_hex_new_class.parquet")

In [ ]:
pois_wide = (
    pois
    .pivot_table(
        index="h3_id",
        columns="category",
        values="count",
        fill_value=0
    )
    .reset_index()
)

In [ ]:
pois_wide

In [ ]:
import pyarrow.parquet as pq

cols_needed = [
    "Origin_Hexagon_ID","Destination_Hexagon_ID", "car_co2"
]

table = pq.read_table("scratch/car_co2_6000.parquet", 
                      columns=cols_needed,
                      use_threads=True)

df_car_co2 = table.to_pandas(types_mapper=pd.ArrowDtype)  # keeps pandas light

In [ ]:
df_car_co2 = df_car_co2.rename(columns={
    "Origin_Hexagon_ID": "from_id",
    "Destination_Hexagon_ID": "to_id",
    "car_co2": "car_co2_total"
})

In [ ]:
df_car_co2_sym = df_car_co2.merge(
    df_car_co2,
    left_on=["from_id", "to_id"],
    right_on=["to_id", "from_id"],
    how="left",
    suffixes=("_outbound", "_inbound")
)


In [ ]:
df_car_co2_sym["car_co2_outbound"] = df_car_co2_sym["car_co2_total_outbound"]

df_car_co2_sym["car_co2_inbound"] = (
    df_car_co2_sym["car_co2_total_inbound"]
    .fillna(df_car_co2_sym["car_co2_total_outbound"])
)

df_car_co2_sym["car_co2_total"] = (
    df_car_co2_sym["car_co2_outbound"]
    + df_car_co2_sym["car_co2_inbound"]
)

In [ ]:
df_car_co2_final = (
    df_car_co2_sym
    .rename(columns={
        "from_id_outbound": "from_id",
        "to_id_outbound": "to_id"
    })[
        [
            "from_id",
            "to_id",
            "car_co2_outbound",
            "car_co2_inbound",
            "car_co2_total"
        ]
    ]
)


In [ ]:
# filter rows where from_id == to_id
same_id_rows = df_car_co2[df_car_co2["from_id"] == df_car_co2["to_id"]]

# show them
same_id_rows

In [ ]:
df_car_co2 = df_car_co2_final.copy()

# value to assign for same-origin/destination trips
car_co2_value = 0.3 * 56 # Walking round trip covering same 

# 1. update existing rows where from_id == to_id
mask_same = df_car_co2["from_id"] == df_car_co2["to_id"]
df_car_co2.loc[mask_same, "car_co2_total"] = car_co2_value

# 2. find which from_ids still need new rows
from_ids = df_car_co2["from_id"].unique()
existing_pairs = set(zip(df_car_co2["from_id"], df_car_co2["to_id"]))

new_rows = [
    {"from_id": fid, "to_id": fid, "car_co2_total": car_co2_value}
    for fid in from_ids
    if (fid, fid) not in existing_pairs
]

# 3. create and append missing rows
df_synthetic_car = pd.DataFrame(new_rows)
df_car_co2 = pd.concat([df_car_co2, df_synthetic_car], ignore_index=True)



In [ ]:
df_car_co2

In [ ]:
# filter rows where from_id == to_id
same_id_rows = df_car_co2[df_car_co2["from_id"] == df_car_co2["to_id"]]

# show them
same_id_rows


In [ ]:
import h3

# 1. get all unique hexes
all_hexes = df_car_co2["from_id"].unique()

# 2. precompute neighbors for each hex (k=1)
neighbors_dict = {
    h: set(h3.k_ring(h, 1)) | {h}  # include self
    for h in all_hexes
}

# 3. vectorized check
df_car_co2["same_or_neighbor"] = df_car_co2.apply(
    lambda row: row["to_id"] in neighbors_dict[row["from_id"]],
    axis=1
)

In [ ]:
df_car_co2.loc[df_car_co2["same_or_neighbor"], "car_co2_total"] = 33.6

In [ ]:
poi_long = (
    pois_wide
    .melt(id_vars='h3_id', var_name='category', value_name='n_pois')
    .query('n_pois > 0')
    [['h3_id', 'category']]
)


In [ ]:
poi_long

In [ ]:
od_home_car = df_car_co2[
    df_car_co2['from_id'].isin(home_users_df['home_gid9'])
]

In [ ]:
od_poi_car = od_home_car.merge(
    poi_long,
    left_on='to_id',
    right_on='h3_id',
    how='inner'
)

In [ ]:
od_poi_car

In [ ]:
nearest_car = (
    od_poi_car
    .sort_values('car_co2_total')
    .groupby(['from_id', 'category'], as_index=False)
    .first()
)

In [ ]:
nearest_wide_car = (
    nearest_car
    .pivot(index='from_id', columns='category', values='car_co2_total')
    .reset_index()
)

In [ ]:
nearest_wide_car

In [ ]:
final_df_car = home_users_df.merge(
    nearest_wide_car,
    left_on='home_gid9',
    right_on='from_id',
    how='left'
)

In [ ]:
final_df_car

In [ ]:
final_df_car = final_df_car.rename(columns={
    'Education_y': 'Education_nearest',
    'Healthcare and Health_y': 'Healthcare and Health_nearest',
    'Others / Not sure_y': 'Others / Not sure_nearest',
    'Recreational, Outdoors_y': 'Recreational, Outdoors_nearest',
    'Shopping, Errands_y': 'Shopping, Errands_nearest',
    'Social, Cultural_y': 'Social, Cultural_nearest'
})

In [ ]:
### Jobs

In [ ]:
import pyarrow.parquet as pq

cols_needed = [
    "Origin_Hexagon_ID", "Destination_Hexagon_ID", "car_co2"
]

table = pq.read_table(
    "scratch/car_co2_6000.parquet", 
    columns=cols_needed,
    use_threads=True
)

df_car_co2 = table.to_pandas(types_mapper=pd.ArrowDtype)  # keeps pandas light

# rename to match OD workflow
df_car_co2 = df_car_co2.rename(columns={
    "Origin_Hexagon_ID": "from_id",
    "Destination_Hexagon_ID": "to_id",
    "car_co2": "car_co2_total"
})

In [ ]:
df_car_co2_sym = df_car_co2.merge(
    df_car_co2,
    left_on=["from_id", "to_id"],
    right_on=["to_id", "from_id"],
    how="left",
    suffixes=("_outbound", "_inbound")
)

In [ ]:
df_car_co2_sym["car_co2_outbound"] = df_car_co2_sym["car_co2_total_outbound"]

df_car_co2_sym["car_co2_inbound"] = (
    df_car_co2_sym["car_co2_total_inbound"]
    .fillna(df_car_co2_sym["car_co2_total_outbound"])
)

df_car_co2_sym["car_co2_total"] = (
    df_car_co2_sym["car_co2_outbound"] + df_car_co2_sym["car_co2_inbound"]
)

In [ ]:
df_car_co2_final = (
    df_car_co2_sym
    .rename(columns={
        "from_id_outbound": "from_id",
        "to_id_outbound": "to_id"
    })[
        [
            "from_id",
            "to_id",
            "car_co2_outbound",
            "car_co2_inbound",
            "car_co2_total"
        ]
    ]
)


In [ ]:
df_car_co2 = df_car_co2_final.copy()

In [ ]:
df_jobs = pd.read_parquet("./data/users_and_works.parquet")

In [ ]:
df_jobs_co2_car = df_jobs.merge(
    df_car_co2,
    left_on=["home_gid9", "work_gid9"],
    right_on=["from_id", "to_id"],
    how="left"
)

In [ ]:
jobs_co2_car = df_jobs_co2_car[['user_id', 'car_co2_total']].rename(
    columns={'car_co2_total': 'job_real_nearest_car'}
)

In [ ]:
final_df_car = final_df_car.merge(
    jobs_co2_car,
    on='user_id',
    how='left'
)


In [ ]:
nearest_cols_car = [c for c in final_df_car.columns if c.endswith('_nearest_car')]

final_df_car = final_df_car.dropna(subset=nearest_cols_car)


In [ ]:
worker_profile = {
    "jobs": 4,
    "Social, Cultural": 2,
    "Shopping, Errands": 1,
    "Recreational, Outdoors": 2
}

In [ ]:
final_df_car

In [ ]:
final_df_car['weekly_co2_expenditure_car'] = (
    worker_profile["jobs"] * final_df_car['job_real_nearest_car'] +
    worker_profile["Social, Cultural"] * final_df_car['Social, Cultural_nearest'] +
    worker_profile["Shopping, Errands"] * final_df_car['Shopping, Errands_nearest'] +
    worker_profile["Recreational, Outdoors"] * final_df_car['Recreational, Outdoors_nearest']
)


In [ ]:
final_df_car.to_parquet("./output/car_expenditure_weekly_nearest.parquet")

In [ ]:
import matplotlib.pyplot as plt

# --- Data ---
df = final_df_car.copy()
df = df[df["weekly_co2_expenditure_car"].notna()]

# Optional: remove extreme outliers for readability
df = df[
    df["weekly_co2_expenditure_car"]
    <= df["weekly_co2_expenditure_car"].quantile(0.99)
]

values = df["weekly_co2_expenditure_car"]

# --- Figure ---
fig, ax = plt.subplots(figsize=(10, 4))

# Boxplot
ax.boxplot(
    values,
    vert=False,
    widths=0.5,
    patch_artist=True,
    boxprops=dict(facecolor="lightgray", edgecolor="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black"),
    flierprops=dict(marker=".", markersize=3, alpha=0.4)
)

# --- Climate budgets ---
ax.axvline(
    7000,
    linestyle="--",
    linewidth=2,
    label="2030 budget (7 kg CO₂ / week)"
)

ax.axvline(
    3000,
    linestyle="--",
    linewidth=2,
    label="2050 budget (3 kg CO₂ / week)"
)

# --- Formatting ---
ax.set_yticks([])
ax.set_xlabel("Weekly CO₂ expenditure per worker (grams)", fontsize=11)
ax.set_title(
    "Minimum Weekly CO₂ Needed to Meet Essential Activities (Car)",
    fontsize=14
)

ax.legend(frameon=False)
ax.grid(axis="x", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()



In [ ]:

import matplotlib.pyplot as plt

# --- Budgets (grams CO₂ / week) ---
BUDGET_2030 = 7000
BUDGET_2050 = 3000

# --- Data ---
df = final_df_car.copy()
df = df[df["weekly_co2_expenditure_car"].notna()]

# optional: trim extreme outliers for readability
df = df[
    df["weekly_co2_expenditure_car"]
    <= df["weekly_co2_expenditure_car"].quantile(0.99)
]

values = df["weekly_co2_expenditure_car"]
total_workers = len(values)

# --- Percentages & counts ---
pct_below_2030 = (values <= BUDGET_2030).mean() * 100
pct_below_2050 = (values <= BUDGET_2050).mean() * 100

count_below_2030 = (values <= BUDGET_2030).sum()
count_above_2030 = (values > BUDGET_2030).sum()

count_below_2050 = (values <= BUDGET_2050).sum()
count_above_2050 = (values > BUDGET_2050).sum()

# --- Figure ---
fig, ax = plt.subplots(figsize=(10, 4))

# Boxplot
ax.boxplot(
    values,
    vert=False,
    widths=0.5,
    patch_artist=True,
    boxprops=dict(facecolor="lightgray", edgecolor="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black"),
    flierprops=dict(marker=".", markersize=3, alpha=0.4)
)

# --- Budget lines with colors ---
ax.axvline(
    BUDGET_2030,
    color="tab:orange",
    linestyle="--",
    linewidth=2,
    label="2030 budget"
)

ax.axvline(
    BUDGET_2050,
    color="tab:blue",
    linestyle="--",
    linewidth=2,
    label="2050 budget"
)

# --- Annotations under plot ---
ax.text(
    0.68, -0.12,
    f"{pct_below_2030:.1f}% below 2030 ({count_below_2030:,})\n"
    f"{count_above_2030:,} above",
    color="tab:orange",
    fontsize=10,
    transform=ax.transAxes,
    va="top"
)

ax.text(
    0.68, -0.24,
    f"{pct_below_2050:.1f}% below 2050 ({count_below_2050:,})\n"
    f"{count_above_2050:,} above",
    color="tab:blue",
    fontsize=10,
    transform=ax.transAxes,
    va="top"
)

# --- Formatting ---
ax.set_yticks([])
ax.set_xlabel("Weekly CO₂ expenditure per worker (grams)", fontsize=11)
ax.set_title(
    "Minimum Weekly CO₂ Needed to Meet Essential Activities (Car)",
    fontsize=14
)

ax.legend(frameon=False)
ax.grid(axis="x", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()



In [ ]:
### Ensure the same user base 